## Retrievers 

A retriever is a component in langchain that fetches relevant documents from a data source in response to a user's query.

There are multiple types of retrievers 
Refer the documentation of langChain

All retrievers in langchain are runnables.

Retrievers can be classified based on two things:
- `Data Source`: Wikipedia retriever , vectorStore retriever , Arxiv retriever
- `Search Strategy`: Maximum marginal relevance (MMR), Multi Query retriever, Contextual compression retriever 



## Data Source retrievers

### **1. Wikipedia Retriever**

It is a retriever that queries the Wikipedia API to fetch relevant content for a given query.


In [2]:
# 1. Install the required Wikipedia and community packages
#%pip install wikipedia langchain-community

from langchain_community.retrievers import WikipediaRetriever

# 2. Initialize the Wikipedia retriever
# top_k_results limits how many matching articles to retrieve
retriever = WikipediaRetriever(top_k_results=2, doc_content_chars_max=500)

# 3. Query Wikipedia directly
query = "Cricket sport"
documents = retriever.invoke(query)

# 4. Print the retrieved contents
for i, doc in enumerate(documents):
    print(f"Result {i+1}:")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}\n")

Result 1:
Metadata: {'title': 'Cricket', 'summary': 'Cricket is a bat-and-ball game that is played between two teams of eleven players on a field, at the centre of which is a 22-yard (20-metre; 66-foot) pitch with a wicket at each end, each comprising two bails (small sticks) balanced on three stumps. Two players from the batting team, the striker and nonstriker, stand in front of either wicket holding bats, while one player from the fielding team, the bowler, bowls the ball toward the striker\'s wicket from the opposite end of the pitch. The striker\'s goal is to hit the bowled ball with the bat and then switch places with the nonstriker, with the batting team scoring one run for each of these swaps. Runs are also scored when the ball reaches the boundary of the field or when the ball is bowled illegally.\nThe fielding team aims to prevent runs by dismissing batters (so they are "out"). Dismissal can occur in various ways, including being bowled (when the ball hits the striker\'s wick

### **2.Vector Store Retreiver**

It is a retriever that queries the Vector DataBase to fetch relevant content for a given query.

In [4]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 1. Prepare sample documents
documents = [
    Document(page_content="Cricket is a bat-and-ball team sport played between two teams."),
    Document(page_content="Python is a versatile programming language used for data science."),
]

# 2. Initialize embeddings (using Gemini or your preferred model)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# 3. Create the vector store
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# 4. Convert the vector store into a retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}  # Returns the top 1 most relevant document
)

# 5. Invoke the retriever with a query
query = "What is Python?"
retrieved_docs = retriever.invoke(query)

# Print results
for i, doc in enumerate(retrieved_docs):
    print(f"Retrieved Doc {i+1}: {doc.page_content}\n")

Retrieved Doc 1: Python is a versatile programming language used for data science.



The above code can be implemented using vector database also but it considers only one metric to organise the similarity where as as_retriever can be used for multiple types

## Search Strategy retrievers

### **1.Maximal Marginal Relevance (MMR)**

MMR is an information retrieval algorithm designed to reduce redundancy in the retrieved results while maintaining high relevance to the query.

It basically picks results that are not only relevant but also different from each other.

It works as follows:
- Picks the most relevant document first
- Then picking the next most relevant and least similar to the selected docs
- and so on..


In [6]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 1. Prepare sample documents
documents = [
    Document(
        page_content=(
            "Cricket is a bat-and-ball team sport played between two teams of"
            " eleven players."
        )
    ),
    Document(
        page_content=(
            "Cricket matches can vary from T20s to five-day Test matches."
        )
    ),
    Document(
        page_content=(
            "Python is a versatile programming language used for data science"
            " and AI."
        )
    ),
]

# 2. Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# 3. Create vector store
vector_store = Chroma.from_documents(documents=documents, embedding=embeddings)

# 4. Configure vector store retriever to use MMR (Maximal Marginal Relevance)
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 2,  # Number of final documents to return
        "fetch_k": 5,  # Number of initial candidate documents to fetch
        "lambda_mult": (
            0.5
        ),  # Balance between relevance (1.0) and diversity (0.0)
    },
)

# 5. Invoke retriever
query = "Tell me about cricket."
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"MMR Result {i+1}: {doc.page_content}")

MMR Result 1: Cricket is a bat-and-ball team sport played between two teams.
MMR Result 2: Cricket matches can vary from T20s to five-day Test matches.


### **2.Multi-Query Retriever**

Sometimes a single query might not capture all the ways information is phrased in your documents.

For example a simple query like " How can i stay healthy? " could mean "what should i eat?","How often should i exercise ?","How can i manage stress?"

It works as follows:
- Takes your original query
- Uses an LLM to generate multiple semantically different versions of that query.
- Performs retrieval for each sub-query.
- Combines and deduplicates the results.

In [11]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 1. Prepare sample documents
documents = [
    Document(page_content="Cricket is a bat-and-ball team sport played between two teams."),
    Document(page_content="The Indian Premier League (IPL) is a major T20 cricket league."),
    Document(page_content="Python is a popular programming language used for data science."),
]

# 2. Initialize embeddings and vector store
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = Chroma.from_documents(documents=documents, embedding=embeddings)

# 3. Initialize your LLM with the updated model name
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

# 4. Create the Multi-Query Retriever
base_retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

# 5. Invoke the retriever
query = "Tell me about cricket tournaments."
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"Multi-Query Result {i+1}: {doc.page_content}")

Multi-Query Result 1: Cricket matches can vary from T20s to five-day Test matches.
Multi-Query Result 2: Cricket matches can vary from T20s to five-day Test matches.
Multi-Query Result 3: The Indian Premier League (IPL) is a major T20 cricket league.
Multi-Query Result 4: The Indian Premier League (IPL) is a major T20 cricket league.
Multi-Query Result 5: The Indian Premier League (IPL) is a major T20 cricket league.


### **3. Contextual Compression Retriever**

It is an advanced Retriever that improves retrieval quality by compressing documents after retrieval

For example a simple query like " What is photosynthesis? " could retrieve an entire paragraph which might not be much relevant where as Contextual Compression Retriever returns only the relevant part.

It works as follows:
- Base retriever retrieves N documents.
- A compressor (usually an LLM) is applied to each document.
- The compressor keeps only parts relevant to the query.
- Irrelevant are discarded.

In [13]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 1. Prepare sample documents
documents = [
    Document(page_content="Cricket is a bat-and-ball team sport played between two teams. The ICC manages international matches globally."),
    Document(page_content="Python is a popular programming language used for data science, machine learning, and backend systems."),
    Document(page_content="The Indian Premier League (IPL) is a major T20 franchise cricket league held annually in India."),
]

# 2. Initialize embeddings and vector store
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = Chroma.from_documents(documents=documents, embedding=embeddings)

# 3. Initialize base retriever and LLM for extraction
base_retriever = vector_store.as_retriever(search_kwargs={"k": 2})
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

# 4. Create the document compressor and contextual compression retriever
compressor = LLMChainExtractor.from_llm(llm)
retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# 5. Invoke retriever (extracts only the relevant sentences/context from chunks)
query = "Tell me about the IPL tournament."
compressed_docs = retriever.invoke(query)

for i, doc in enumerate(compressed_docs):
    print(f"Compressed Result {i+1}: {doc.page_content}")

Compressed Result 1: The Indian Premier League (IPL) is a major T20 cricket league.
Compressed Result 2: The Indian Premier League (IPL) is a major T20 cricket league.


For more retrievers refer :
https://docs.langchain.com/oss/python/integrations/retrievers